In [1]:
import pandas as pd
from pandas import read_csv, read_json
import numpy as np

In [3]:
raw_csv = read_csv('../data/raw/reunion_segments.csv')
df = pd.DataFrame(raw_csv)
raw_parquet = pd.read_parquet('../data/raw/reunion_segments.parquet')
df_parquet = pd.DataFrame(raw_parquet)

In [8]:
df['gradient'] = (df['elevation_gain'] / df['distance']) * 100  # %
df['avg_speed'] = df['distance'] / df['best_time']  # m/s
df['avg_speed_kmh'] = df['avg_speed'] * 3.6  # km/h


In [ ]:
df_ride = df[df['activity_type'] == 'Ride'].copy()
df_run = df[df['activity_type'] == 'Run'].copy()

### Ride

In [ ]:
def uphill_factor(gradient, distance):
    if 10 < gradient <= 13 and distance > 15:
        return 4
    elif 8 < gradient <= 10 and distance > 30:
        return 4.5
    elif 5 < gradient <= 8 and distance > 60:
        return 5
    elif 3 < gradient <= 5 and distance > 120:
        return 6
    else:
        return 7

In [ ]:
def slope_factor(gradient, distance):
    if gradient <-30:
        return 1.5
    elif -30< gradient <0:
        return 1 + 2*0.7/13*gradient +0.7/13**2*gradient**2
    elif 0<= gradient <20:
        return 1 + (gradient/uphill_factor(gradient,distance))**2
    else:
        return 10

In [6]:
def expected_road_speed(gradient, distance):
    base_speed = 40/3.6  # m/s on flat road
    return base_speed / slope_factor(gradient, distance)

In [9]:
df['expected_road_speed'] = df.apply(lambda row: expected_road_speed(row['gradient'], row['distance']), axis=1)

# Speed ratio: actual / expected
# < 1 means slower than road = rough terrain
df['speed_ratio'] = df['avg_speed_kmh'] / df['expected_road_speed']

# === Consistency ratio ===
# Technical terrain = more variance in times
df['consistency_ratio'] = df['best_time'] / df['average_top_10_time']
# Closer to 1 = consistent (road), lower = high variance (technical)

### Run

## Power profile

In [2]:
def power_profile_function(distance_km):
    power = np.zeros_like(distance_km)
    mask_short = distance_km <= 1
    mask_long = distance_km > 1

    power[mask_short] = 800 * np.exp(-0.425 * distance_km[mask_short]) 
    power[mask_long] = np.exp(-distance_km[mask_long]/50+5.8) + 200

    return power

In [4]:
import numpy as np
import plotly.graph_objects as go

x_points = np.array([15, 30, 60, 2*60, 3*60, 5*60, 10*60, 15*60, 20*60, 30*60, 45*60, 60*60])
y_points = np.array([1083, 978, 606, 536, 489, 419, 417, 407, 341, 319, 300, 276])

# Distance (on évite 0 pour le log)
distance = np.linspace(0.1, 100, 1000)
power = power_profile_function(distance)

fig = go.Figure()
"""
fig.add_trace(go.Scatter(
    x=distance,
    y=power,
    mode="lines",
    name="Power vs Distance"
))
"""
fig.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Power",
    #xaxis_type="log",        # Axe logarithmique
    template="plotly_white"
)
fig.add_scatter(x=x_points, y=y_points, mode='markers', name='Data Points', marker=dict(color='red', size=8))

fig.show()


In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit

x_points = np.array([15, 30, 60, 2*60, 3*60, 5*60, 10*60, 15*60, 20*60, 30*60, 45*60, 60*60])
y_points = np.array([1083, 978, 606, 536, 489, 419, 417, 407, 341, 319, 300, 276])

# Séparer les données en deux plages
mask_short = x_points < 300
x_short = x_points[mask_short]
y_short = y_points[mask_short]

mask_long = x_points >= 300
x_long = x_points[mask_long]
y_long = y_points[mask_long]

# Fonction puissance pour les courtes durées (modèle classique de power-duration)
def power_short(t, CP, W_prime):
    """Modèle hyperbolique: P = CP + W'/t"""
    return CP + W_prime / t**(1/2)

# Fonction logarithmique pour les longues durées
def power_long(t, a, b, c):
    """Modèle logarithmique: P = a - b * log(t - c)"""
    return a - b * np.log(t - c)

# Fit pour les durées courtes (15-300s)
params_short, _ = curve_fit(power_short, x_short, y_short, p0=[400, 10000])
CP, W_prime = params_short
print(f"Short duration fit (15-300s):")
print(f"  CP (Critical Power) = {CP:.1f} W")
print(f"  W' (Anaerobic capacity) = {W_prime:.1f} J")

# Fit pour les durées longues (300s+)
params_long, _ = curve_fit(power_long, x_long, y_long, p0=[450, 50, 200])
a, b, c = params_long
print(f"\nLong duration fit (300s+):")
print(f"  a = {a:.1f}, b = {b:.1f}, c = {c:.1f}")

# Créer la fonction composite
def power_profile_function(t):
    """Fonction composite qui utilise les deux fits"""
    result = np.zeros_like(t)
    mask_short = t < 300
    mask_long = t >= 300
    
    result[mask_short] = power_short(t[mask_short], CP, W_prime)
    result[mask_long] = power_long(t[mask_long], a, b, c)
    
    return result

# Générer les courbes
time = np.linspace(15, 3600, 1000)
power = power_profile_function(time)

# Visualisation
fig = go.Figure()

# Courbe complète
fig.add_trace(go.Scatter(
    x=time,
    y=power,
    mode="lines",
    name="Power Profile (composite)",
    line=dict(color='blue', width=2)
))

# Fit court (en pointillés pour voir la distinction)
time_short = np.linspace(15, 300, 100)
fig.add_trace(go.Scatter(
    x=time_short,
    y=power_short(time_short, CP, W_prime),
    mode="lines",
    name="Short fit (15-300s)",
    line=dict(color='green', dash='dash'),
    visible='legendonly'
))

# Fit long (en pointillés)
time_long = np.linspace(300, 3600, 100)
fig.add_trace(go.Scatter(
    x=time_long,
    y=power_long(time_long, a, b, c),
    mode="lines",
    name="Long fit (300s+)",
    line=dict(color='orange', dash='dash'),
    visible='legendonly'
))

# Points de données originaux
fig.add_scatter(
    x=x_points, 
    y=y_points, 
    mode='markers', 
    name='Data Points', 
    marker=dict(color='red', size=8)
)

# Ligne verticale à 300s pour montrer la transition
fig.add_vline(x=300, line_dash="dot", line_color="gray", 
              annotation_text="Transition (300s)")

fig.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Power (W)",
    template="plotly_white",
    hovermode='x unified'
)

fig.show()

# Vérifier la qualité du fit
residuals_short = y_short - power_short(x_short, CP, W_prime)
residuals_long = y_long - power_long(x_long, a, b, c)
rmse_short = np.sqrt(np.mean(residuals_short**2))
rmse_long = np.sqrt(np.mean(residuals_long**2))

print(f"\nQuality of fit:")
print(f"  RMSE short: {rmse_short:.1f} W")
print(f"  RMSE long: {rmse_long:.1f} W")

Short duration fit (15-300s):
  CP (Critical Power) = 220.2 W
  W' (Anaerobic capacity) = 3522.0 J

Long duration fit (300s+):
  a = 1274.3, b = 119.4, c = -870.8



Quality of fit:
  RMSE short: 63.5 W
  RMSE long: 15.4 W


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit

x_points = np.array([15, 30, 60, 2*60, 3*60, 5*60, 10*60, 15*60, 20*60, 30*60, 45*60, 60*60])
y_points = np.array([1083, 978, 606, 536, 489, 419, 417, 407, 341, 319, 300, 276])

# Modèle généralisé de power-duration
def power_profile(t, P_inf, AWC, alpha):
    """
    Modèle généralisé de profil de puissance
    P(t) = P∞ + AWC / t^alpha
    
    Args:
        P_inf: Puissance asymptotique (puissance théorique à durée infinie) (W)
        AWC: Anaerobic Work Capacity - Capacité de travail anaérobie (W·s^alpha)
        alpha: Exposant de décroissance (typiquement ~0.5 pour √t)
    """
    return P_inf + AWC / t**alpha

def power_profile_2(duration_seconds):
    """
    Calcule le profil de puissance en fonction de la durée (en secondes)
    en utilisant une logique vectorisée (np.where) pour gérer les tableaux NumPy.
    """

    # Paramètres du modèle (Définis comme des constantes NumPy pour le calcul vectoriel)
    CP = 250.0      # Critical Power (W)
    W_prime = 20000.0 # Réserve anaérobie en Joules (20 kJ)
    P_max = 1000.0  # Puissance maximale sur 5 secondes
    
    # Assurez-vous que l'entrée est un tableau (même si c'est un seul nombre)
    duration_seconds = np.array(duration_seconds, dtype=float)
    
    # Initialisation du tableau de résultats à CP par défaut
    power = CP * np.ones_like(duration_seconds)

    # --- Logique vectorisée avec np.where ---
    
    # 1. Cas : duration_seconds < 5 secondes (Sprint très court: P_max)
    # Si la condition est VRAIE, utilise P_max, sinon utilise la valeur dans 'power'
    power = np.where((duration_seconds < 5) & (duration_seconds > 0), P_max,power)

    # 2. Cas : 5 <= duration_seconds < 300 secondes (Utilisation de W')
    # Attention: cette étape DOIT écraser les valeurs dans la plage [5, 300[
    power = np.where((duration_seconds >= 5) & (duration_seconds < 300),CP + (W_prime / duration_seconds),power)

    # 3. Cas : duration_seconds >= 300 secondes (Au-delà de 5 min: proche de CP)
    power = np.where(duration_seconds >= 300,CP * (1 + 50.0 / duration_seconds),power)
    
    # 4. Cas : duration_seconds <= 0
    power = np.where(duration_seconds <= 0, 0.0,power)

    return power
# Fit du modèle
params, _ = curve_fit(power_profile, x_points, y_points, p0=[200, 10000, 0.5])
P_inf, AWC, alpha = params

print(f"📊 Power Profile Model:")
print(f"  P∞ (Asymptotic Power) = {P_inf:.1f} W")
print(f"  AWC (Anaerobic Work Capacity) = {AWC:.1f} W·s^{alpha:.2f}")
print(f"  α (decay exponent) = {alpha:.3f}")
print(f"\n💡 Interpretation:")
print(f"  - P∞ is the theoretical sustainable power at infinite duration")
print(f"  - With α ≈ 0.5, this follows the classic √t decay model")
print(f"  - AWC quantifies the anaerobic reserve for short efforts")
print(f"  - Actual FTP (1h power) ≈ {power_profile(3600, P_inf, AWC, alpha):.0f} W")

# Générer les courbes
time = np.linspace(15, 3600, 1000)
power_fitted = power_profile(time, P_inf, AWC, alpha)

# Calculer quelques points de référence
power_1min = power_profile(60, P_inf, AWC, alpha)
power_5min = power_profile(300, P_inf, AWC, alpha)
power_20min = power_profile(1200, P_inf, AWC, alpha)
power_1h = power_profile(3600, P_inf, AWC, alpha)

print(f"\n⏱️ Reference Powers:")
print(f"  1 min:  {power_1min:.0f} W")
print(f"  5 min:  {power_5min:.0f} W")
print(f"  20 min: {power_20min:.0f} W")
print(f"  1 hour (FTP): {power_1h:.0f} W")
print(f"  P∞:     {P_inf:.0f} W")

# Visualisation
fig = go.Figure()

# Courbe complète
fig.add_trace(go.Scatter(
    x=time,
    y=power_fitted,
    mode="lines",
    name=f"Power Profile (α={alpha:.2f})",
    line=dict(color='blue', width=2)
))

# Seconde courbe
fig.add_trace(go.Scatter(
    x=time,
    y=power_profile(time, 250, 2700, 0.5),
    mode="lines",
    name=f"Power Profile 2 ",
    line=dict(color='green', width=2)
))

# Points de données originaux
fig.add_scatter(
    x=x_points, 
    y=y_points, 
    mode='markers', 
    name='Measured Data', 
    marker=dict(color='red', size=10, symbol='circle')
)

# Ajouter des lignes de référence pour les durées clés
reference_times = [60, 300, 1200, 3600]
reference_labels = ['1min', '5min', '20min', '1h']
for t, label in zip(reference_times, reference_labels):
    if t <= 3600:
        p = power_profile(t, P_inf, AWC, alpha)
        fig.add_vline(x=t, line_dash="dot", line_color="gray", opacity=0.5,
                      annotation_text=f"{label}: {p:.0f}W", 
                      annotation_position="top")

fig.update_layout(
    title="Power Duration Curve",
    xaxis_title="Duration (seconds)",
    yaxis_title="Power (Watts)",
    template="plotly_white",
    hovermode='x unified',
    showlegend=True
)

fig.show()

# Qualité du fit
residuals = y_points - power_profile(x_points, P_inf, AWC, alpha)
rmse = np.sqrt(np.mean(residuals**2))
r_squared = 1 - (np.sum(residuals**2) / np.sum((y_points - np.mean(y_points))**2))

print(f"\n📈 Model Quality:")
print(f"  RMSE: {rmse:.1f} W")
print(f"  R²: {r_squared:.4f}")
print(f"  Max residual: {np.max(np.abs(residuals)):.1f} W")

📊 Power Profile Model:
  P∞ (Asymptotic Power) = 238.7 W
  AWC (Anaerobic Work Capacity) = 3330.2 W·s^0.49
  α (decay exponent) = 0.491

💡 Interpretation:
  - P∞ is the theoretical sustainable power at infinite duration
  - With α ≈ 0.5, this follows the classic √t decay model
  - AWC quantifies the anaerobic reserve for short efforts
  - Actual FTP (1h power) ≈ 298 W

⏱️ Reference Powers:
  1 min:  684 W
  5 min:  441 W
  20 min: 341 W
  1 hour (FTP): 298 W
  P∞:     239 W



📈 Model Quality:
  RMSE: 46.1 W
  R²: 0.9660
  Max residual: 113.1 W


: 